In [ ]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [ ]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [ ]:
compact = True

In [ ]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

In [ ]:
raw_df.dtypes

In [ ]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [ ]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

In [ ]:
df.dtypes

## Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [ ]:
X

In [ ]:
y

In [ ]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [ ]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

In [ ]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

In [ ]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

In [ ]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

### Attempt to infer card probabilities from rank and table

In [ ]:
from cpp_poker.cpp_poker import Hand, CardCollection, HandRank, CheatSheet

In [ ]:
hand_ranks_per_state = []
table_ranks = []
for i, (state_id, row) in enumerate(prob_df.iterrows()):
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    table_rank = table_cards.rank_hand().get_rank()
    excess_hand_ranks_row = table_cards.rank_all_hands()
    hand_ranks_per_state.append(
        [rank.get_rank() - table_rank for rank in excess_hand_ranks_row]
    )
    table_ranks.append(table_rank)

table_ranks_df = pd.DataFrame(table_ranks, index=prob_df.index, columns=["table_rank"])
hand_ranks_df = pd.DataFrame(hand_ranks_per_state, index=prob_df.index)
hand_ranks_df

In [ ]:
# Get the values of the column indices from hand_ranks_df
column_indices = hand_ranks_df.values

# Use np.arange to create an array of row indices
row_indices = np.arange(hand_ranks_df.shape[0])[:, None]

# Use the row and column indices to index into the DataFrame values
hand_probs_df = pd.DataFrame(
    prob_df.values[row_indices, column_indices],
    index=hand_ranks_df.index,
    columns=hand_ranks_df.columns,
)

# Normalize the hand probabilities to sum to 1 for each row
hand_probs_df = hand_probs_df.div(hand_probs_df.sum(axis=1), axis=0)
hand_probs_df

In [ ]:
# Validate that rows sum to 1
hand_probs_df.sum(axis=1)

In [ ]:
# Look at the max probabilites for different hands
mask = prob_df["pred"] == 2
max_probs = hand_probs_df[mask].max(axis=1)
max_prob_hands = hand_probs_df[mask].idxmax(axis=1)
sample_prob_df = pd.DataFrame({"max_prob": max_probs, "hand": max_prob_hands})
sample_prob_df["hand"] = sample_prob_df["hand"].apply(lambda x: Hand(x).str())
sample_prob_df["table"] = raw_df.loc[sample_prob_df.index, "public_cards"].apply(
    lambda x: CardCollection(x).str()
)
sample_prob_df["predicted_excess_rank"] = prob_df.loc[sample_prob_df.index, "pred"]
sample_prob_df["pred_rank"] = sample_prob_df["predicted_excess_rank"] + table_ranks_df.loc[
    sample_prob_df.index, "table_rank"
]
sample_prob_df["pred_rank_str"] = sample_prob_df["pred_rank"].apply(
    lambda x: HandRank(x, []).get_rank_name()
)
sample_prob_df = sample_prob_df.drop(["predicted_excess_rank"], axis=1)
sample_prob_df

### Attempt to infer winning probabilities from hidden state probabilities

In [ ]:
hand_winning_probs = []
for i, (state_id, row) in enumerate(hand_probs_df.iterrows()):
    print(f"Processing row {i}\n")
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    processed_row = df.loc[state_id]
    n_players = processed_row["n_players"]
    hand_winning_probs.append(CheatSheet.get_all_winning_probabilities(table_cards, n_players, 1000))

hand_winning_probs_df = pd.DataFrame(hand_winning_probs, index=hand_probs_df.index)
hand_winning_probs_df